# Solution: Simple PID Controller


In [13]:
# Import standard libraries
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [14]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
LEFT_WHEEL_POS = "left_wheel_pos"
RIGHT_WHEEL_POS = "right_wheel_pos"
LEFT_WHEEL_VEL = "left_wheel_vel"
RIGHT_WHEEL_VEL = "right_wheel_vel"

In [15]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [16]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [17]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [18]:
# Filter and PID coefficients (tune these)
ALPHA = 0.99
KP = 15.0      # Proportional constant
KD = 0.5      # Derivative constant

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
pitch = 0.0
tipped = False
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Define free-look camera (control with mouse), looking at robot's back-right
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8 
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        # Read sensors
        accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
        pitch_rate = data.sensor(IMU_GYRO).data[1]

        # Accelerometer-derived pitch
        accel_pitch = -1 * math.atan2(accel_x, accel_z)

        # Copmlementary filter to estimate pitch from accelerometer and gyroscope
        pitch = ALPHA * (pitch + (pitch_rate * model.opt.timestep)) + \
                (1 - ALPHA) * accel_pitch

        # Uncomment to use ground-truth pitch (from orientation) instead of estimated pitch
        # w, x, y, z = data.sensor(IMU_ORIENTATION).data
        # pitch = math.asin(max(-1.0, min(1.0, 2.0 * (w * y - z * x))))

        # If the robot tips over, stop the wheels
        if abs(pitch) > TIP_THRESHOLD:
            tipped = True
        
        if tipped:
            motor_spd_target = 0.0
        else:
            # PD control (no I term needed)
            motor_spd_target = clamp((KP * pitch) + (KD * pitch_rate))

        # Update motor speeds
        data.ctrl[left_motor_id] = motor_spd_target
        data.ctrl[right_motor_id] = motor_spd_target

        # Advance simulation by one step (determined by timestep attribute in MJCF file)
        mujoco.mj_step(model, data)

        # Render the current simulation state
        viewer.sync()

        # Print the feedback values every PRINT_EVERY steps
        steps += 1
        if steps % PRINT_EVERY == 0:
            clear_output(wait=True)
            print(f"Pitch:              {pitch} rad")
            print(f"Pitch rate:         {pitch_rate} rad/s")
            print(f"Motor speed target: {motor_spd_target}")
        
        # Make sure the loop iteration time matches the MJCF timestep time (2 ms)
        # i.e. slow the simulation down so we can see the robot behave in real time
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

Pitch:              0.004790284332180609 rad
Pitch rate:         -0.09312439656726773 rad/s
Motor speed target: 0.025292066699075272
